# Preprocessing Pipeline (ordered flow)

**Input:** the per-participant EEG files written by `build.py --export-eeg` (RiseTogether-analysis repo), in `out/eeg/`.

**How to use:** run the notebook **once per participant**. Set `SESSION` and `IPAD` in the parameters cell, inspect the plots, fill in that participant's bad channels and (if needed) ICA components, then run the rest.
Every output is written to a separate `derivatives` folder; the raw files are never modified.

**Flow:** 1. load → 2. events → 3. crop + mark breaks → 4. resample → 5. bad channels → 6. ICA → 7. filter + apply ICA → 8. interpolate → 9. re-reference → 10. save + report

Requirements: `uv sync`

## Creating the EEG files with build.py

Run in the RiseTogether-analysis repo (see its `docs/students/` guide). Read `out/qa_report.md` after every build.

In [ ]:
"""
Run in the terminal, from the RiseTogether-analysis folder:

uv run build.py --data-root 01_raw_data --out out --export-eeg   # real sessions (listed in sessions.toml)
uv run build.py --data-root 01_raw_data --out out --export-eeg --adhoc-session   # practice/test data

Writes one set of files per participant (headset) to out/eeg/:
    s<session>_<ipad>_<hsd stem>_eeg_raw.fif   32 channels, 500 Hz, events as annotations  <- used here
    s<session>_<ipad>_<hsd stem>_emg_raw.fif   EMG, 2000 Hz (only the first 4 channels are used)
    s<session>_<ipad>_<hsd stem>_imu_raw.fif   motion sensors
"""

Loading the packages

In [1]:
import mne
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from mne.preprocessing import ICA
from mne_icalabel import label_components
%matplotlib inline

/Users/yoavlevy/Bsc/riseup_preproccesing/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 0. Parameters

All settings live here, to be reported in Methods.

In [ ]:
# --- Participant to process (run the notebook once per participant) ---
SESSION = 1  # TODO: session number, as in sessions.toml
IPAD = "pia"  # TODO: the iPad (= participant) of this run: pia, patrick, polly, padmé, peter or parsnip
SUBJECT = f"s{SESSION}_{IPAD}"  # used to name all outputs

# --- Paths ---
RT_ANALYSIS_DIR = Path("~/Bsc/RiseTogether-analysis").expanduser()  # the RiseTogether-analysis repo
DATA_DIR = RT_ANALYSIS_DIR / "out" / "eeg"  # the .fif files from build.py --export-eeg
CAP_LAYOUT = RT_ANALYSIS_DIR / "explore" / "cap_layout.csv"  # hardware channel (p0_ch1) -> electrode (O1)
OUT_DIR = RT_ANALYSIS_DIR / "out" / "derivatives" / f"sub-{SUBJECT}"  # out/ is ignored by git there
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Channels ---
MONTAGE = "colin27_1020"  
EOG_CHANNELS = ["EOG2", "EOG3", "EOG4"]  # from cap_layout.csv 
# M1/M2 (mastoids) are kept as EEG channels: they are on the head and have standard positions

# Bad channels per participant, filled in after inspecting the data in step 5
BAD_CHANNELS = {
    "s1_pia": [],
}

# --- Events ---
CONDITIONS = ["motor_baseline", "individual", "joint"]  # trials to keep; the rest of the session is cropped/marked as break
TRIAL_START = "TRIAL_START/"  # e.g. TRIAL_START/joint/r1/t2
TRIAL_END = "TRIAL_END/"
SYNC_ANNOTATIONS = r"^(trig_in\d|GPIO_TRIGGER/|PHOTON/)"  # used by build.py for alignment only; removed here
BREAK_MIN_DURATION = 15.0  # gaps between trials longer than this (s) are marked as BAD_break

# --- Filtering, resampling and reference ---
SFREQ = 250  # Hz, plenty for alpha and beta, and makes ICA faster
ICA_L_FREQ, ICA_H_FREQ = 1.0, 100.0  # copy used only for ICA (ICLabel expects 1-100 Hz, average reference)
L_FREQ, H_FREQ = 0.1, 40.0  # analysis data
REFERENCE = "average"  # TODO: confirm with the lab; mastoids would be ["M1", "M2"]

# --- ICA ---
ICA_N_COMPONENTS = 0.95  # components explaining 95% of the variance
ICA_REJECT = dict(eeg=200e-6)  # skip 2 s segments with peak-to-peak > 200 µV when fitting ICA
ICLABEL_ARTIFACTS = ["eye blink", "muscle artifact", "heart beat", "line noise", "channel noise"]
ICLABEL_THRESHOLD = 0.8  # exclude a component if ICLabel is at least 80% sure it is an artifact
# After visual review, the final list of components to exclude per participant
# (None = use the automatic selection from step 6)
ICA_EXCLUDE_MANUAL = {
    "s1_pia": None,
}
RANDOM_STATE = 97  # fixed seed so ICA gives the same result every run

## 1. Loading the raw data

In [ ]:
# One EEG file per participant; the last part of the name (the .hsd recording) is not known in advance
files = sorted(DATA_DIR.glob(f"s{SESSION}_{IPAD}_*_eeg_raw.fif"))
assert len(files) == 1, f"expected one EEG file for {SUBJECT}, found {[f.name for f in files]}"
RAW_FILE = files[0]

# preload=True is needed for filtering, re-referencing and ICA
raw = mne.io.read_raw(RAW_FILE, preload=True)
print(f"Subject: {SUBJECT}, Channels: {raw.info['nchan']}, Duration: {raw.times[-1]:.2f} s, Sampling Rate: {raw.info['sfreq']} Hz")

# The files use hardware channel names (p0_ch1 ...); rename them to electrode names
cap_layout = pd.read_csv(CAP_LAYOUT)
raw.rename_channels(dict(zip(cap_layout["channel"], cap_layout["label"])))

# Channel types: EOG channels must not be treated as EEG
raw.set_channel_types({ch: "eog" for ch in EOG_CHANNELS})

# Set montage (electrode positions, needed for ICLabel, topographies and interpolation)
raw.set_montage(MONTAGE)


## 2. Events

`build.py` already placed every event as an **annotation** at the right sample of this headset, so they stay correct through cropping, resampling and every later step.
Each participant's own events are used: in the individual condition each iPad runs its own game, so the trial windows differ slightly between participants.

In [ ]:
# A headset that build.py could not align has no annotations: its data cannot be used
assert len(raw.annotations) > 0, f"{SUBJECT}: no annotations, this headset could not be aligned (see qa_report.md)"

# Remove the sync annotations
is_sync = pd.Series(raw.annotations.description).str.match(SYNC_ANNOTATIONS).to_numpy()
raw.annotations.delete(np.where(is_sync)[0])
print(f"Removed {is_sync.sum()} sync annotations, kept {len(raw.annotations)}")

# Trials found in this file
trial_starts = pd.Series([d for d in raw.annotations.description if d.startswith(TRIAL_START)])
print(trial_starts.str.split("/").str[1].value_counts())

events, event_id = mne.events_from_annotations(raw, regexp="^TRIAL_(START|END)/")
fig = mne.viz.plot_events(events, event_id=event_id, sfreq=raw.info["sfreq"], first_samp=raw.first_samp)

### Check: do the files of all participants look complete?

Run once all files of the session exist. Every participant should have the same number of trials per condition.
The files do **not** start at the same moment (the headsets were switched on one at a time), so do not compare start times or lengths: the annotations are the alignment.

In [ ]:
rows = []
for path in sorted(DATA_DIR.glob(f"s{SESSION}_*_eeg_raw.fif")):
    r = mne.io.read_raw(path, preload=False, verbose="error")
    starts = [d.split("/")[1] for d in r.annotations.description if d.startswith(TRIAL_START)]
    row = {"file": path.name, "sfreq": r.info["sfreq"], "n_channels": r.info["nchan"], "aligned": len(r.annotations) > 0}
    row.update(pd.Series(starts, dtype=str).value_counts().to_dict())
    rows.append(row)
pd.DataFrame(rows)

## 3. Cropping to the task and marking breaks

The recording covers the whole session (welcome, surveys, video, debriefing). ICA should only learn from task data:
setup, instructions and long breaks contain movement and talking that do not occur during play.
`CONDITIONS` sets which trials are kept. The motor baseline is kept by default: everyone sees the same bubbles at the same moment without interacting, so it can serve as a control for coupling that comes only from shared stimuli.

In [ ]:
def trial_times(raw, prefix):
    """Times (s, from the start of the data) of the TRIAL_START/ or TRIAL_END/ annotations of the kept conditions."""
    pattern = "^" + prefix + "(" + "|".join(CONDITIONS) + ")/"
    ev, _ = mne.events_from_annotations(raw, regexp=pattern, verbose="error")
    return np.sort((ev[:, 0] - raw.first_samp) / raw.info["sfreq"])


starts, ends = trial_times(raw, TRIAL_START), trial_times(raw, TRIAL_END)

# Mark long gaps between the end of a trial and the start of the next one (surveys, results screens,
# instructions, team reassignment) as BAD_break, so ICA and later steps ignore them.
# Note for RQ3: the results screen (who won) comes after a trial; if those moments are needed,
# they must not be marked as break.
# This is done BEFORE cropping: the files have no meas_date, and MNE would shift all events
# by the cropped amount if annotations were added after a crop.
break_onsets, break_durations = [], []
for end, next_start in zip(ends[:-1], starts[1:]):
    if next_start - end > BREAK_MIN_DURATION:
        break_onsets.append(end + 1.0)
        break_durations.append(next_start - end - 2.0)  # 1 s margin on each side

# Annotation onsets are counted from the measurement start, so add the time of the first sample
onset_offset = raw.first_time if raw.annotations.orig_time is not None else 0.0
breaks = mne.Annotations(
    onset=np.array(break_onsets) + onset_offset,
    duration=break_durations,
    description="BAD_break",
    orig_time=raw.annotations.orig_time,
)
raw.set_annotations(raw.annotations + breaks)

# Keep only the task: from the first trial start to the last trial end, with a 2 s margin
raw.crop(tmin=max(starts.min() - 2.0, 0), tmax=min(ends.max() + 2.0, raw.times[-1]))
print(f"Breaks marked: {len(breaks)}, total {np.sum(break_durations):.1f} s of {raw.times[-1]:.1f} s")

## 4. Downsampling to 250 Hz

Done early (on the continuous data, not on epochs) to make every later step faster. The annotations are kept automatically.

In [ ]:
raw.resample(sfreq=SFREQ)

## 5. Identifying bad channels

Bad channels are different for every participant. Inspect the data, then write this participant's bad channels into `BAD_CHANNELS` in the parameters cell.
Tip: with `%matplotlib qt` you can click on channels in `raw.plot()` to mark them as bad.

They are **not** interpolated yet: ICA is fitted without them, and they are interpolated after ICA (step 8).

In [ ]:
raw.plot(n_channels=30, scalings=dict(eeg=50e-6))
raw.compute_psd(fmax=60).plot()

In [ ]:
raw.info["bads"] = BAD_CHANNELS.get(SUBJECT, [])
print("Bad channels:", raw.info["bads"])

## 6. ICA

**Labeling:** ICLabel classifies each component; EOG correlation is used as a second opinion. Always check the automatic choice visually.

In [ ]:
# Filtering the data for ICA training
raw_ica = raw.copy().filter(l_freq=ICA_L_FREQ, h_freq=ICA_H_FREQ)
raw_ica.set_eeg_reference("average")  # bad channels are left out of the average automatically

ica = ICA(
    n_components=ICA_N_COMPONENTS,
    method="infomax",
    fit_params=dict(extended=True),  # extended Infomax, which ICLabel was trained on
    max_iter="auto",
    random_state=RANDOM_STATE,
)
ica.fit(raw_ica, picks="eeg", reject=ICA_REJECT, reject_by_annotation=True)
print(ica)

In [ ]:
# Automatic labeling with ICLabel
ic_labels = label_components(raw_ica, ica, method="iclabel")
components = pd.DataFrame({
    "component": range(ica.n_components_),
    "label": ic_labels["labels"],
    "probability": ic_labels["y_pred_proba"].round(2),
})
is_artifact = components["label"].isin(ICLABEL_ARTIFACTS) & (components["probability"] >= ICLABEL_THRESHOLD)

# Second opinion: components that correlate with the EOG channels
eog_present = [ch for ch in EOG_CHANNELS if ch in raw_ica.ch_names]
eog_indices = []
if eog_present:
    eog_indices, eog_scores = ica.find_bads_eog(raw_ica, ch_name=eog_present)
components["eog_correlated"] = components["component"].isin(eog_indices)

auto_exclude = sorted(set(components.loc[is_artifact, "component"]) | set(eog_indices))
print("Automatically selected for exclusion:", auto_exclude)
components

In [ ]:
# Visual check of the components
ica.plot_components()
ica.plot_sources(raw_ica, show_scrollbars=False)
if auto_exclude:
    ica.plot_properties(raw_ica, picks=auto_exclude)

In [ ]:
manual = ICA_EXCLUDE_MANUAL.get(SUBJECT)
ica.exclude = [int(i) for i in (auto_exclude if manual is None else manual)]
print("Excluded components:", ica.exclude)

# Save the ICA solution and the component table (to report in the Methods)
components["excluded"] = components["component"].isin(ica.exclude)
ica.save(OUT_DIR / f"sub-{SUBJECT}_ica.fif", overwrite=True)
components.to_csv(OUT_DIR / f"sub-{SUBJECT}_ica_components.csv", index=False)

## 7. Filtering the analysis data and applying ICA

The ICA weights from step 6 are applied to the analysis data (0.1–40 Hz).
It is set to average reference first, because ICA was fitted on average-referenced data.
No notch filter is needed: the 40 Hz low-pass already removes the 50 Hz line noise.

In [ ]:
raw_clean = raw.copy().filter(l_freq=L_FREQ, h_freq=H_FREQ)
raw_clean.set_eeg_reference("average")

raw_before_ica = raw_clean.copy()
ica.apply(raw_clean)
ica.plot_overlay(raw_before_ica, picks="eeg")  # before vs. after removing the components

## 8. Interpolating bad channels

In [ ]:
bad_channels = list(raw_clean.info["bads"])  # keep the list for the summary
raw_clean.interpolate_bads(reset_bads=True)

## 9. Re-referencing

Done after interpolation, so the interpolated channels are part of the reference.

In [ ]:
raw_clean.set_eeg_reference(REFERENCE)
raw_clean.plot(n_channels=30, scalings=dict(eeg=50e-6))

## 10. Saving the cleaned data and a quality report

The report (HTML) shows the data before and after cleaning and the ICA components. The summary table collects the numbers to report in the Methods for all participants.

In [ ]:
raw_clean.save(OUT_DIR / f"sub-{SUBJECT}_clean_raw.fif", overwrite=True)

report = mne.Report(title=f"Preprocessing sub-{SUBJECT}")
report.add_raw(raw, title="Before cleaning (cropped, resampled)", psd=True)
report.add_ica(ica, title="ICA", inst=raw_ica)
report.add_raw(raw_clean, title="After cleaning", psd=True)
report.save(OUT_DIR / f"sub-{SUBJECT}_report.html", overwrite=True, open_browser=False)

# One row per participant; re-running a participant replaces its row
summary_file = OUT_DIR.parent / "preprocessing_summary.csv"
row = pd.DataFrame([{
    "subject": SUBJECT,
    "bad_channels": " ".join(bad_channels),
    "n_bad_channels": len(bad_channels),
    "n_ica_components": ica.n_components_,
    "excluded_components": " ".join(map(str, ica.exclude)),
    "n_excluded_components": len(ica.exclude),
    "break_s": round(float(np.sum(break_durations)), 1),
    "duration_s": round(raw_clean.times[-1], 1),
}])
if summary_file.exists():
    previous = pd.read_csv(summary_file, dtype={"subject": str})
    row = pd.concat([previous[previous["subject"] != SUBJECT], row])
row.to_csv(summary_file, index=False)
row

## Next: analysis (separate notebook)

The ERP epoching from the original notebook (−0.2 to 0.5 s around events) was removed, because it does not fit the cRQA analysis. The next steps work on the cleaned continuous data:

1. Band-pass alpha (8–12 Hz) and beta (13–30 Hz) and compute the amplitude envelope on the data (cutting first would add filter edge artifacts to every segment).
2. Cut the envelopes into trials using the `TRIAL_START/` and `TRIAL_END/` annotations.
3. **Cut every participant's data relative to the same event, never relative to the file start.** 
4. **Teams change between joint blocks:** take each pair's teammate/opponent label per trial from `team_id` in the `trials` table.
5. Average over ROIs, downsample the envelopes to ~20–50 Hz, then run cRQA.